<a href="https://colab.research.google.com/github/CristianoMSilva/DistilBERT-FL-Playground/blob/main/03_baseline_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Check the GPU of this session (recorded with the results)
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

GPU available: True
GPU: Tesla T4
GPU memory (GB): 15.6


In [2]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
df = load_dataset("zefang-liu/phishing-email-dataset")["train"].to_pandas()

# Clean (same steps as 01_eda.ipynb)
df = df.drop(columns=["Unnamed: 0"])
df = df.dropna(subset=["Email Text"])
conflicts = df.groupby("Email Text")["Email Type"].nunique()
df = df[~df["Email Text"].isin(conflicts[conflicts > 1].index)]
df = df.drop_duplicates(subset="Email Text")
word_count = df["Email Text"].str.split().str.len()
df = df[(word_count > 0) & (word_count < 1_000_000)].reset_index(drop=True)

# Labels: 1 = phishing, 0 = safe
df["label"] = (df["Email Type"] == "Phishing Email").astype(int)

# Stratified split: 80% train, 10% validation, 10% test (same as 02_tokenization.ipynb)
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(part)} emails, {part['label'].mean():.1%} phishing")

README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv: reconstructing file:   0%|          |  0.00B / 52.0MB            

Phishing_Email.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

Train: 14027 emails, 37.4% phishing
Validation: 1753 emails, 37.4% phishing
Test: 1754 emails, 37.4% phishing


In [8]:
from datasets import Dataset
from transformers import AutoTokenizer

# Settings in one place: change MAX_LENGTH to 512 later for the comparison run
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Convert a DataFrame to the Hugging Face format used in training
def to_dataset(part):
    return Dataset.from_pandas(part[["Email Text", "label"]].reset_index(drop=True))

# Tokenize, cutting emails longer than MAX_LENGTH tokens
def tokenize(batch):
    return tokenizer(batch["Email Text"], truncation=True, max_length=MAX_LENGTH)

train_ds = to_dataset(train_df).map(tokenize, batched=True, remove_columns=["Email Text"])
val_ds = to_dataset(val_df).map(tokenize, batched=True, remove_columns=["Email Text"])
test_ds = to_dataset(test_df).map(tokenize, batched=True, remove_columns=["Email Text"])

print(train_ds)

Map:   0%|          | 0/14027 [00:00<?, ? examples/s]

Map:   0%|          | 0/1753 [00:00<?, ? examples/s]

Map:   0%|          | 0/1754 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 14027
})


In [9]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Class weights: errors on the minority class (phishing) cost more
class_weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_df["label"])
print("Class weights (safe, phishing):", class_weights.round(3))

# Metrics with phishing (label 1) as the positive class
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1)
    return {"accuracy": accuracy_score(labels, preds), "precision": precision, "recall": recall, "f1": f1}

Class weights (safe, phishing): [0.799 1.337]


In [10]:
import torch
from torch import nn
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

# Weighted loss: uses the class weights from cell 4
weights = torch.tensor(class_weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(weight=weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

# Builds a fresh model + trainer (used for the quick test and the full run)
def build_trainer(train_data, eval_data, epochs, output_dir):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=True,
        logging_steps=100,
        seed=42,
        report_to="none",
    )
    return WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

print("Ready")

Ready


In [11]:
import time

# Quick test: 500 training emails, 200 validation emails, 1 epoch
smoke = build_trainer(train_ds.select(range(500)), val_ds.select(range(200)),
                      epochs=1, output_dir="smoke_test")

start = time.time()
smoke.train()
print(f"Training time: {time.time() - start:.0f} s")
print(smoke.evaluate())

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.522249,0.930000,0.868421,0.942857,0.904110


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training time: 12 s


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
No log,0.522249,1,0.930000,0.868421,0.942857,0.904110


{'eval_loss': 0.5222494006156921, 'eval_accuracy': 0.93, 'eval_precision': 0.868421052631579, 'eval_recall': 0.9428571428571428, 'eval_f1': 0.9041095890410958}


In [12]:
# Full training: all 14,027 training emails, 3 epochs, max_length 256
trainer = build_trainer(train_ds, val_ds, epochs=3, output_dir="results_256")

start = time.time()
trainer.train()
train_time = time.time() - start
print(f"Training time: {train_time / 60:.1f} min")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.091335,0.044576,0.986309,0.975867,0.987786,0.981791
2,0.028634,0.043233,0.991443,0.987805,0.989313,0.988558
3,0.003671,0.052646,0.990873,0.987786,0.987786,0.987786


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training time: 6.0 min


In [13]:
from sklearn.metrics import confusion_matrix

# Final evaluation on the test set (best epoch, never seen during training)
test_output = trainer.predict(test_ds)
print({k: round(v, 4) for k, v in test_output.metrics.items() if k.startswith("test_")})

# Confusion matrix and false positive rate
preds = np.argmax(test_output.predictions, axis=-1)
tn, fp, fn, tp = confusion_matrix(test_output.label_ids, preds).ravel()
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"False positive rate: {fp / (fp + tn):.2%}")

{'test_loss': 0.0475, 'test_accuracy': 0.9863, 'test_precision': 0.9847, 'test_recall': 0.9787, 'test_f1': 0.9817, 'test_runtime': 4.1749, 'test_samples_per_second': 420.126, 'test_steps_per_second': 13.174}
TP=642  FP=10  FN=14  TN=1088
False positive rate: 0.91%


In [14]:
# Collect test metrics, false positive rate and training time of a run
def summarize(output, train_time):
    preds = np.argmax(output.predictions, axis=-1)
    tn, fp, fn, tp = confusion_matrix(output.label_ids, preds).ravel()
    m = output.metrics
    return {"accuracy": m["test_accuracy"], "precision": m["test_precision"],
            "recall": m["test_recall"], "f1": m["test_f1"],
            "false_positive_rate": fp / (fp + tn), "train_time_min": train_time / 60}

# Save the max_length 256 results before training again
results = {256: summarize(test_output, train_time)}
print(results)

{256: {'accuracy': 0.9863169897377423, 'precision': 0.9846625766871165, 'recall': 0.9786585365853658, 'f1': 0.981651376146789, 'false_positive_rate': np.float64(0.009107468123861567), 'train_time_min': 6.043005303541819}}


In [15]:
# Same emails, now keeping up to 512 tokens per email
MAX_LENGTH = 512
train_ds = to_dataset(train_df).map(tokenize, batched=True, remove_columns=["Email Text"])
val_ds = to_dataset(val_df).map(tokenize, batched=True, remove_columns=["Email Text"])
test_ds = to_dataset(test_df).map(tokenize, batched=True, remove_columns=["Email Text"])

# Train a fresh model with the same settings
trainer = build_trainer(train_ds, val_ds, epochs=3, output_dir="results_512")
start = time.time()
trainer.train()
train_time = time.time() - start

# Test and save the results
test_output = trainer.predict(test_ds)
results[512] = summarize(test_output, train_time)
print(results[512])

Map:   0%|          | 0/14027 [00:00<?, ? examples/s]

Map:   0%|          | 0/1753 [00:00<?, ? examples/s]

Map:   0%|          | 0/1754 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.080768,0.035478,0.988021,0.981763,0.986260,0.984006
2,0.032025,0.041180,0.989732,0.986260,0.986260,0.986260
3,0.007746,0.039785,0.992584,0.990826,0.989313,0.990069


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'accuracy': 0.991448118586089, 'precision': 0.9908116385911179, 'recall': 0.9862804878048781, 'f1': 0.988540870893812, 'false_positive_rate': np.float64(0.00546448087431694), 'train_time_min': 11.16991218328476}


In [1]:
import pandas as pd

# Test results copied from the outputs of cells 9 and 10 (runtime disconnected after cell 10)
results = {
    256: {"accuracy": 0.9863, "precision": 0.9847, "recall": 0.9787, "f1": 0.9817,
          "false_positive_rate": 0.0091, "train_time_min": 6.0},
    512: {"accuracy": 0.9914, "precision": 0.9908, "recall": 0.9863, "f1": 0.9885,
          "false_positive_rate": 0.0055, "train_time_min": 11.2},
}

# Comparison table: one row per max_length
table = pd.DataFrame(results).T
table.index.name = "max_length"
table

,accuracy,precision,recall,f1,false_positive_rate,train_time_min
max_length,,,,,,
256,0.9863,0.9847,0.9787,0.9817,0.0091,6.0
512,0.9914,0.9908,0.9863,0.9885,0.0055,11.2
